Imports

In [2]:
import os
import mediapipe as mp
import cv2
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings

warnings.filterwarnings('ignore')

Set up Hands

In [3]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
hands = mp_hands.Hands(static_image_mode = True, min_detection_confidence = 0.1, max_num_hands=1)

I0000 00:00:1749105584.935284 3043043 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1 Pro


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1749105584.946463 3043300 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1749105584.953738 3043300 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Hands on images

In [16]:
def visualize_data(data_dir):

    for label in sorted(os.listdir(data_dir)):
        label_path = os.path.join(data_dir, label)
        if not os.path.isdir(label_path):
            continue
            
        for img_file in os.listdir(label_path)[:1]:
            image_path = os.path.join(label_path, img_file)
            img = cv2.imread(image_path)
            if img is None:
                continue  # Skip corrupt/invalid images
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = hands.process(img_rgb)
            
            # Skip if no hand or multiple hands detected
            if not results.multi_hand_landmarks or len(results.multi_hand_landmarks) != 1:
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Detect hands
            results = hands.process(img_rgb)
            
            if results.multi_hand_landmarks:
                # Draw landmarks on original image (BGR for saving)
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        img_rgb,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                        mp_drawing_styles.get_default_hand_landmarks_style(),
                        mp_drawing_styles.get_default_hand_connections_style()
                    )
        plt.figure()
        plt.title(image_path)
        plt.imshow(img_rgb)
    plt.show()

In [1]:
data_train1 = 'ASL_Dataset/Train'
data_test1 = 'ASL_Dataset/Test'

data_train2 = 'Lexset/Train'
data_test2 = 'Lexset/Test'

#visualize_data(data_train1)

Create data to learn

In [4]:
def load_data(data_dir):
    
    data = []
    labels = []
    
    for label in sorted(os.listdir(data_dir)):
        label_path = os.path.join(data_dir, label)
        if not os.path.isdir(label_path):
            continue
            
        for img_file in os.listdir(label_path):
            image_path = os.path.join(label_path, img_file)
            img = cv2.imread(image_path)
            if img is None:
                continue  # Skip corrupt/invalid images
            
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = hands.process(img_rgb)
            
            # Skip if no hand or multiple hands detected
            if not results.multi_hand_landmarks or len(results.multi_hand_landmarks) != 1:
                continue
            
            # Extract landmarks (21 points, x/y normalized coordinates)
            hand_landmarks = results.multi_hand_landmarks[0]
            positions = []
            for landmark in hand_landmarks.landmark:
                positions.extend([landmark.x, landmark.y, landmark.z])  # 42 features per image
            
            data.append(positions)
            labels.append(label)
    
    # Save data with purpose (e.g., "Train" or "Test")
    purpose = os.path.basename(data_dir)
    base_dir = os.path.basename(os.path.dirname(data_dir))
    save_file = f'data_3D/{base_dir}_{purpose}.pkl'
    
    with open(save_file, 'wb') as f:
        pickle.dump({'data': data, 'labels': labels}, f)
    
    #return data, labels

In [5]:
load_data(data_test1)
load_data(data_test2)
load_data(data_train1)
load_data(data_train2)

W0000 00:00:1749105597.330347 3043300 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


ML Classifier

In [6]:
def load_pickle_data(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return np.array(data['data']), np.array(data['labels'])

In [7]:
X_train1, y_train1 = load_pickle_data('data_3D/ASL_Dataset_Train.pkl')
X_test1, y_test1 = load_pickle_data('data_3D/ASL_Dataset_Test.pkl')

X_train2, y_train2 = load_pickle_data('data_3D/Lexset_Train.pkl')
X_test2, y_test2 = load_pickle_data('data_3D/Lexset_Test.pkl')

# Concatenate training data
X_train = np.concatenate([X_train1, X_train2], axis=0)
y_train = np.concatenate([y_train1, y_train2], axis=0)

# Concatenate test data
X_test = np.concatenate([X_test1, X_test2], axis=0)
y_test = np.concatenate([y_test1, y_test2], axis=0)

#Verify Data Shapes
print(f"Training data shape: {X_train.shape}, Labels shape: {y_train.shape}")
print(f"Testing data shape: {X_test.shape}, Labels shape: {y_test.shape}")

#Train Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=22,
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)

print("\nModel Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

with open('random_forest_model_lexasl_3D.pkl', 'wb') as f:
    pickle.dump(model, f)


Training data shape: (48303, 63), Labels shape: (48303,)
Testing data shape: (2557, 63), Labels shape: (2557,)

Model Performance:
Accuracy: 0.99
Classification Report:
              precision    recall  f1-score   support

           A       0.99      0.98      0.99       101
           B       1.00      1.00      1.00        97
           C       0.99      1.00      0.99        99
           D       0.97      0.95      0.96        92
           E       0.98      0.97      0.97        95
           F       1.00      0.99      0.99        97
           G       1.00      1.00      1.00       103
           H       1.00      1.00      1.00       102
           I       1.00      1.00      1.00        97
           J       0.95      0.98      0.97       101
           K       0.99      0.99      0.99        98
           L       1.00      0.99      0.99       100
           M       0.99      0.98      0.98        95
           N       0.96      1.00      0.98       100
           O       0

Real-Time Performance

In [ ]:
with open('random_forest_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

In [6]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        continue
        
    # Convert BGR to RGB and process
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)
    
    # Draw hand landmarks
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )
    
    # Display frame
    cv2.imshow('ASL Detection', frame)
    
    # Exit on 'q' press
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1748563539.476558 16523990 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
